# Classificação supervisionada com CART

Este roteiro usa os polígonos salvos como assets no Earth Engine para treinar e validar uma classificação binária:

- `landcover = 0`: floresta;
- `landcover = 1`: agricultura.

Fluxo: inicializar → criar o feature space → carregar e rotular os polígonos → sobrepor os pixels → separar treino/validação → treinar CART → classificar → avaliar.

In [1]:
import ee

from repository_ee import (
    GeometryUtils,
    Landsat8,
    LocalLabels,
    MapLayers,
    Models,
    PeriodoAgricola,
    initialize_earth_engine,
)

## 1. Inicialização e parâmetros gerais

A AOI e o ano precisam ser os mesmos usados na criação das amostras.

In [12]:
initialize_earth_engine()

ANO = 2025
ESCALA = 30
SEED = 42
FRACAO_TREINO = 0.70
AMOSTRAS_POR_CLASSE = 1000
PROPRIEDADE_CLASSE = "landcover"

COORDS_TOCANTINS = [
    [-48.46995289774661, -10.859497023647087],
    [-47.84922535868411, -10.859497023647087],
    [-47.84922535868411, -10.362094203375317],
    [-48.46995289774661, -10.362094203375317],
    [-48.46995289774661, -10.859497023647087],
]

aoi = GeometryUtils.polygon(COORDS_TOCANTINS)
periodo = PeriodoAgricola.tocantins(ANO)

## 2. Imagem e variáveis preditoras

O `feature_space` reúne as bandas Landsat e as métricas temporais calculadas pelo repositório. A mesma lista e ordem de bandas será usada no treinamento e na classificação.

In [3]:
landsat = Landsat8.collection(aoi, periodo)
feature_space = Landsat8.feature_space(landsat, aoi, periodo)
bandas_preditoras = feature_space.bandNames()

print("Quantidade de bandas preditoras:", bandas_preditoras.size().getInfo())
print("Bandas:", bandas_preditoras.getInfo())

Quantidade de bandas preditoras: 98
Bandas: ['blue', 'green', 'red', 'nir', 'swir1', 'swir2', 'temp', 'ndvi', 'evi2', 'cai', 'ndwi', 'gcvi', 'hall_cover', 'pri', 'blue_dry_dry', 'green_dry_dry', 'red_dry_dry', 'nir_dry_dry', 'swir1_dry_dry', 'swir2_dry_dry', 'temp_dry_dry', 'ndvi_dry_dry', 'evi2_dry_dry', 'cai_dry_dry', 'ndwi_dry_dry', 'gcvi_dry_dry', 'hall_cover_dry_dry', 'pri_dry_dry', 'blue_wet_wet', 'green_wet_wet', 'red_wet_wet', 'nir_wet_wet', 'swir1_wet_wet', 'swir2_wet_wet', 'temp_wet_wet', 'ndvi_wet_wet', 'evi2_wet_wet', 'cai_wet_wet', 'ndwi_wet_wet', 'gcvi_wet_wet', 'hall_cover_wet_wet', 'pri_wet_wet', 'blue_stdDev', 'green_stdDev', 'red_stdDev', 'nir_stdDev', 'swir1_stdDev', 'swir2_stdDev', 'temp_stdDev', 'ndvi_stdDev', 'evi2_stdDev', 'cai_stdDev', 'ndwi_stdDev', 'gcvi_stdDev', 'hall_cover_stdDev', 'pri_stdDev', 'blue_min', 'green_min', 'red_min', 'nir_min', 'swir1_min', 'swir2_min', 'temp_min', 'ndvi_min', 'evi2_min', 'cai_min', 'ndwi_min', 'gcvi_min', 'hall_cover_min', 'pr

## 3. Carregamento e rotulagem dos polígonos

O campo `landcover` é definido novamente com base no asset de origem. Isso é intencional: os dois arquivos podem conter o mesmo `class_id` usado durante o desenho.

In [4]:
ASSET_FLORESTA = (
    "projects/poetic-tesla-506814-e1/assets/tocantins_2025_FLORESTA"
)
ASSET_AGRICULTURA = (
    "projects/poetic-tesla-506814-e1/assets/tocantins_2025_AGRICULTURA"
)

poligonos_floresta = ee.FeatureCollection(ASSET_FLORESTA).map(
    lambda feature: feature.set(PROPRIEDADE_CLASSE, 0).set("classe_nome", "FLORESTA")
)
poligonos_agricultura = ee.FeatureCollection(ASSET_AGRICULTURA).map(
    lambda feature: feature.set(PROPRIEDADE_CLASSE, 1).set("classe_nome", "AGRICULTURA")
)

poligonos = poligonos_floresta.merge(poligonos_agricultura).filterBounds(aoi)

print("Polígonos de floresta:", poligonos_floresta.size().getInfo())
print("Polígonos de agricultura:", poligonos_agricultura.size().getInfo())
print("Distribuição dos polígonos:", Models.class_distribution(
    poligonos, PROPRIEDADE_CLASSE
).getInfo())

Polígonos de floresta: 14
Polígonos de agricultura: 65
Distribuição dos polígonos: {'0': 14, '1': 65}


## 4. Rasterização dos polígonos e sobreposição à imagem

Os polígonos são convertidos em uma banda `landcover`, mascarada fora das áreas rotuladas. Em seguida, `Models.stratified_sample()` extrai uma quantidade controlada e balanceada de pixels por classe. Isso evita materializar todos os pixels dos polígonos e exceder o limite do Earth Engine.

In [13]:
labels = LocalLabels.from_feature_collection(
    features=poligonos,
    class_property=PROPRIEDADE_CLASSE,
    geometry=aoi,
    reference_image=feature_space,
    band_name=PROPRIEDADE_CLASSE,
)

amostras = Models.stratified_sample(
    feature_space=feature_space,
    labels=labels,
    class_band=PROPRIEDADE_CLASSE,
    region=aoi,
    num_points=AMOSTRAS_POR_CLASSE,
    class_values=[0, 1],
    class_points=[AMOSTRAS_POR_CLASSE, AMOSTRAS_POR_CLASSE],
    scale=ESCALA,
    seed=SEED,
    tile_scale=2,
    geometries=True,
)

distribuicao = Models.class_distribution(amostras, PROPRIEDADE_CLASSE).getInfo()
if len(distribuicao) < 2:
    raise ValueError(
        f"São necessárias duas classes após a sobreposição; recebido: {distribuicao}"
    )

print("Pixels amostrados por classe:", distribuicao)
print("Total de pixels amostrados:", amostras.size().getInfo())

Pixels amostrados por classe: {'0': 1000, '1': 1000}
Total de pixels amostrados: 2000


In [15]:
map = MapLayers.create(aoi, zoom=9)
map.addLayer(amostras, {"color": "1a9850", "fillColor": "1a985066"}, "Amostras")
map

Map(center=[-10.61088033468291, -48.15958912821537], controls=(WidgetControl(options=['position', 'transparent…

## 5. Separação entre treinamento e validação

A coluna aleatória é reproduzível por causa do `SEED`. A validação não participa do treinamento.

In [16]:
amostras_treino, amostras_validacao = Models.split_samples(
    amostras,
    training_fraction=FRACAO_TREINO,
    seed=SEED,
)

distribuicao_treino = Models.class_distribution(
    amostras_treino, PROPRIEDADE_CLASSE
).getInfo()
distribuicao_validacao = Models.class_distribution(
    amostras_validacao, PROPRIEDADE_CLASSE
).getInfo()

if len(distribuicao_treino) < 2 or len(distribuicao_validacao) < 2:
    raise ValueError(
        "Treino e validação precisam conter floresta e agricultura. "
        "Desenhe mais polígonos ou altere o SEED."
    )

print("Treinamento:", distribuicao_treino)
print("Validação:", distribuicao_validacao)

Treinamento: {'0': 689, '1': 710}
Validação: {'0': 311, '1': 290}


## 6. Treinamento do CART

A criação e o treinamento do classificador passam pela classe `Models`. `max_nodes` controla o tamanho máximo da árvore e `min_leaf_population` evita folhas sustentadas por pouquíssimos pixels.

In [17]:
modelo_cart = Models.train(
    samples=amostras_treino,
    input_properties=bandas_preditoras,
    class_property=PROPRIEDADE_CLASSE,
    algorithm="cart",
    classifier_parameters={
        "max_nodes": 50,
        "min_leaf_population": 5,
    },
)

modelo_cart.explain()

## 7. Classificação da imagem

A imagem é classificada com exatamente as mesmas bandas, na mesma ordem, usadas pelo treinamento.

In [18]:
classificacao_cart = Models.classify_image(
    feature_space=feature_space,
    classifier=modelo_cart,
    input_properties=bandas_preditoras,
    output_band=PROPRIEDADE_CLASSE,
).clip(aoi)

VIS_CART = {
    "bands": [PROPRIEDADE_CLASSE],
    "min": 0,
    "max": 1,
    "palette": ["006400", "ffcc00"],
}

mapa = MapLayers.create(aoi, zoom=9)
MapLayers.add_rgb(mapa, feature_space, "Landsat — RGB")
mapa.addLayer(classificacao_cart, VIS_CART, "CART — floresta x agricultura")
mapa.addLayer(poligonos_floresta, {"color": "006400"}, "Polígonos — floresta")
mapa.addLayer(poligonos_agricultura, {"color": "ffcc00"}, "Polígonos — agricultura")
mapa

Map(center=[-10.61088033468291, -48.15958912821537], controls=(WidgetControl(options=['position', 'transparent…

## 8. Avaliação do treinamento e validação independente

A matriz de treinamento mede o ajuste aos dados vistos pelo CART. A matriz de validação é a referência principal, pois usa os pixels separados anteriormente.

In [20]:
metricas_treino = Models.training_metrics(modelo_cart)
metricas_validacao = Models.validation_metrics(
    classifier=modelo_cart,
    validation_samples=amostras_validacao,
    class_property=PROPRIEDADE_CLASSE,
    prediction_property="classification",
    class_order=[0, 1],
)

print("Matriz de confusão — treinamento:")
print(metricas_treino["confusion_matrix"].getInfo())
print("Acurácia de treinamento:", metricas_treino["accuracy"].getInfo())

print("\nMatriz de confusão — validação:")
print(metricas_validacao["confusion_matrix"].getInfo())
print("Acurácia de validação:", metricas_validacao["accuracy"].getInfo())
print("Kappa de validação:", metricas_validacao["kappa"].getInfo())
print("Acurácia do produtor:", metricas_validacao["producers_accuracy"].getInfo())
print("Acurácia do consumidor:", metricas_validacao["consumers_accuracy"].getInfo())

Matriz de confusão — treinamento:
[[688, 1], [5, 705]]
Acurácia de treinamento: 0.995711222301644

Matriz de confusão — validação:
[[308, 3], [5, 285]]
Acurácia de validação: 0.9866888519134775
Kappa de validação: 0.9733389524115827
Acurácia do produtor: [[0.9903536977491961], [0.9827586206896551]]
Acurácia do consumidor: [[0.9840255591054313, 0.9895833333333334]]


## Observação metodológica

Este roteiro faz a divisão aleatória após extrair os pixels, adequada para uma primeira verificação. Para uma avaliação espacial mais rigorosa, separe polígonos inteiros ou regiões geográficas antes da extração, evitando que pixels vizinhos do mesmo polígono apareçam simultaneamente no treino e na validação.